In [5]:
import cv2
import datetime
import sys
import os
import glob
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from tqdm import tqdm
# from ConvNeXt_model import ConvNeXt
import pytz

# from sklearn.metrics import roc_curve, roc_auc_score


from pathlib import Path
import seaborn as sns
import timm
from pprint import pprint
# from Mammodataset import Mammo_datasets
# from cbloss import CBLoss

class before_datasets(Dataset):
    def __init__(self, dir, transform = None):
        self.transform = transform
        self.dir = dir
        self.data = []
        self.labels = []
        name_label = {"0" : 0, "1" : 1}

        target_dir = os.path.join(dir,"*/*.jpg")

        for path in glob.glob(target_dir):
            name = path.split("\\")[3]
            label = name_label[name]

            self.data.append(path)
            self.labels.append(label)
                
    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img_path = self.data[index]
        label = self.labels[index]
       # file_name = os.path.basename(img_path)

        img = cv2.imread(img_path, -1)

        if self.transform:
            img = self.transform(img)

        return img, label

epochs = 200
lr = 0.0001
BATCH_SIZE =128

train_path = "D:\\Yamato\\before_train2"
val_path = "D:\\Yamato\\before_val2"

transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),  # すべての画像をサイズ (224, 224) にリサイズする
    transforms.Normalize(mean=[0.5], std=[0.5])
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),  # すべての画像をサイズ (224, 224) にリサイズする
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# データセットを定義
train = before_datasets(dir=train_path, transform=transform_train)
val = before_datasets(dir=val_path, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = torch.utils.data.DataLoader(val, batch_size=BATCH_SIZE, shuffle=False)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

# model = timm.create_model(model_name='convnext_tiny.fb_in22k_ft_in1k', pretrained=True, in_chans=1, num_classes=2)
model = timm.create_model(model_name='resnet200d.ra2_in1k', pretrained=True, in_chans=1, num_classes=2)

model.to(device)




import timm
from torch.nn import Sequential, Dropout,Module
import numpy as np

train_acc_list = []
val_acc_list = []
train_loss_list = []
val_loss_list = []
best_accuracy = 0
m = nn.Sigmoid()
n = nn.Softmax(dim = 1)
best_accuracy = 0
    
# # 各クラスのデータ数
# class_counts = torch.tensor([1568, 833])  # クラス0: 1000件, クラス1: 200件
# total_count = class_counts.sum()
# # 重みを計算
# class_weights = 1.0 / class_counts.float()
# class_weights = class_weights / class_weights.sum()  # 正規化 (オプション)
# print(class_weights)    
# criterion = nn.CrossEntropyLoss(weight=class_weights).to(device)

# positive_weight = 2.0  # 負例に比べて正例が10倍少ない場合
# criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight))
# criterion = nn.CrossEntropyLoss(weight=torch.tensor(positive_weight)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

# optimizer = optim.AdamW(model.parameters(), lr=lr,weight_decay=1e-4)
    

for epoch in range(epochs):
    epoch_loss = 0
    epoch_accuracy = 0

    model.train()

    for data, label in tqdm(train_loader):
        data = data.to(device)
        label = label.to(device)

        output = model(data)
        output = n(output)
        loss = criterion(output, label)

        optimizer.zero_grad()
        loss.backward() 
        optimizer.step()

        acc = (output.argmax(dim=1) == label).float().mean()
        epoch_accuracy += acc / len(train_loader)
        epoch_loss += loss / len(train_loader)

    model.eval()

    with torch.no_grad():
        epoch_val_accuracy = 0
        epoch_val_loss = 0

        for data, label in valid_loader:
            data = data.to(device)
            label = label.to(device)

            val_output = model(data)
            val_output = n(val_output)
            val_loss = criterion(val_output, label)

            acc = (val_output.argmax(dim=1) == label).float().mean()
            epoch_val_accuracy += acc / len(valid_loader)
            epoch_val_loss += val_loss / len(valid_loader)

        if epoch_val_accuracy > best_accuracy:
            torch.save(model.state_dict(), f'ResNet200_konpe_before.pth')
            best_accuracy = epoch_val_accuracy
            
    print(f"Epoch : {epoch+1} - loss : {epoch_loss:.4f} - acc : {epoch_accuracy:.4f} - val_loss : {epoch_val_loss:.4f} - val_acc : {epoch_val_accuracy:.4f}\n")
    train_acc_list.append(epoch_accuracy)
    train_loss_list.append(epoch_loss)
    val_acc_list.append(epoch_val_accuracy)
    val_loss_list.append(epoch_val_loss)

model_path = f'D:\\Python\\ResNet200_konpe_before.pth'
model.load_state_dict(torch.load(model_path))

epochs = 200
lr = 0.0001
BATCH_SIZE =128

train_path = "D:\\Yamato\\train2"
val_path = "D:\\Yamato\\valid2"

transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),  # すべての画像をサイズ (224, 224) にリサイズする
    transforms.Normalize(mean=[0.5], std=[0.5])
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),  # すべての画像をサイズ (224, 224) にリサイズする
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# データセットを定義
train = before_datasets(dir=train_path, transform=transform_train)
val = before_datasets(dir=val_path, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = torch.utils.data.DataLoader(val, batch_size=BATCH_SIZE, shuffle=False)

import timm
from torch.nn import Sequential, Dropout,Module
import numpy as np

train_acc_list = []
val_acc_list = []
train_loss_list = []
val_loss_list = []
best_accuracy = 0
m = nn.Sigmoid()
n = nn.Softmax(dim = 1)
best_accuracy = 0
    
# # 各クラスのデータ数
# class_counts = torch.tensor([1568, 833])  # クラス0: 1000件, クラス1: 200件
# total_count = class_counts.sum()
# # 重みを計算
# class_weights = 1.0 / class_counts.float()
# class_weights = class_weights / class_weights.sum()  # 正規化 (オプション)
# print(class_weights)    
# criterion = nn.CrossEntropyLoss(weight=class_weights).to(device)

# positive_weight = 2.0  # 負例に比べて正例が10倍少ない場合
# criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight))
# criterion = nn.CrossEntropyLoss(weight=torch.tensor(positive_weight)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

# optimizer = optim.AdamW(model.parameters(), lr=lr,weight_decay=1e-4)
    

for epoch in range(epochs):
    epoch_loss = 0
    epoch_accuracy = 0

    model.train()

    for data, label in tqdm(train_loader):
        data = data.to(device)
        label = label.to(device)

        output = model(data)
        output = n(output)
        loss = criterion(output, label)

        optimizer.zero_grad()
        loss.backward() 
        optimizer.step()

        acc = (output.argmax(dim=1) == label).float().mean()
        epoch_accuracy += acc / len(train_loader)
        epoch_loss += loss / len(train_loader)

    model.eval()

    with torch.no_grad():
        epoch_val_accuracy = 0
        epoch_val_loss = 0

        for data, label in valid_loader:
            data = data.to(device)
            label = label.to(device)

            val_output = model(data)
            val_output = n(val_output)
            val_loss = criterion(val_output, label)

            acc = (val_output.argmax(dim=1) == label).float().mean()
            epoch_val_accuracy += acc / len(valid_loader)
            epoch_val_loss += val_loss / len(valid_loader)

        if epoch_val_accuracy > best_accuracy:
            torch.save(model.state_dict(), f'ResNet200_konpe.pth')
            best_accuracy = epoch_val_accuracy
            
    print(f"Epoch : {epoch+1} - loss : {epoch_loss:.4f} - acc : {epoch_accuracy:.4f} - val_loss : {epoch_val_loss:.4f} - val_acc : {epoch_val_accuracy:.4f}\n")
    train_acc_list.append(epoch_accuracy)
    train_loss_list.append(epoch_loss)
    val_acc_list.append(epoch_val_accuracy)
    val_loss_list.append(epoch_val_loss)

# import os
# import cv2
# import numpy as np
# from tqdm import tqdm

# def is_inverted(image, threshold=128):
#     """
#     画像が反転しているかどうかを判定する関数
#     - 平均輝度値で判断する
#     """
#     mean_intensity = np.mean(image)
#     # 平均輝度が高い場合、画像が反転しているとみなす
#     return mean_intensity > threshold

# def process_images(input_dir, output_dir, threshold=128):
#     """
#     画像を統一する関数
#     - 反転している画像を修正し、指定のフォルダに保存
#     """
#     os.makedirs(output_dir, exist_ok=True)
#     for file_name in tqdm(os.listdir(input_dir)):
#         file_path = os.path.join(input_dir, file_name)
        
#         # 画像を読み込み（グレースケールで読み取る）
#         image = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
#         if image is None:
#             print(f"Failed to read image: {file_path}")
#             continue
        
#         # 反転しているか判定
#         if is_inverted(image, threshold):
#             # 反転を修正（ピクセル値を反転）
#             image = 255 - image
        
#         # 統一された画像を保存
#         output_path = os.path.join(output_dir, file_name)
#         cv2.imwrite(output_path, image)

# # 入力ディレクトリと出力ディレクトリを指定
# input_dir = "D:\\Python\\BreastCancer\\before_train\\1"  # 元の画像が保存されているフォルダ
# output_dir = "D:\\Python\\BreastCancer\\before_train1\\1"  # 修正後の画像を保存するフォルダ

# # 閾値を設定して実行
# process_images(input_dir, output_dir, threshold=128)

# import os
# import cv2
# import numpy as np
# from tqdm import tqdm

# def detect_breast_orientation(image):
#     """
#     乳房の向きを検出する関数
#     - 右向きなら "right" を返し、左向きなら "left" を返す
#     """
#     # 画像の形状を取得
#     h, w = image.shape
    
#     # 左側と右側の輝度合計を計算
#     left_sum = np.sum(image[:, :w // 2])
#     right_sum = np.sum(image[:, w // 2:])
    
#     # 左右の輝度を比較して向きを判定
#     if left_sum > right_sum:
#         return "left"
#     else:
#         return "right"

# def process_images(input_dir, output_dir, target_orientation="left"):
#     """
#     マンモグラフィ画像の向きを統一する関数
#     - 指定した向き（target_orientation）に全ての画像を統一
#     """
#     os.makedirs(output_dir, exist_ok=True)
#     for file_name in tqdm(os.listdir(input_dir)):
#         file_path = os.path.join(input_dir, file_name)
        
#         # グレースケールで画像を読み込む
#         image = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
#         if image is None:
#             print(f"Failed to read image: {file_path}")
#             continue
        
#         # 乳房の向きを検出
#         orientation = detect_breast_orientation(image)
        
#         # 向きを修正（必要に応じて反転）
#         if orientation != target_orientation:
#             image = cv2.flip(image, 1)  # 左右反転（1: 左右、0: 上下）
        
#         # 修正後の画像を保存
#         output_path = os.path.join(output_dir, file_name)
#         cv2.imwrite(output_path, image)

# # 入力ディレクトリと出力ディレクトリを指定
# input_dir = "D:\\Python\\BreastCancer\\before_train1\\1"  # 元の画像が保存されているフォルダ
# output_dir = "D:\\Python\\BreastCancer\\before_train2\\1"  # 修正後の画像を保存するフォルダ

# # 向きを「左向き」に統一
# process_images(input_dir, output_dir, target_orientation="left")


# import cv2
# import numpy as np
# import os
# from tqdm import tqdm

# def extract_breast_region(image):
#     """
#     マンモグラフィ画像から乳房領域を切り出す関数
#     """
#     # グレースケール化（すでにグレースケールの場合、この行は不要）
#     gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    
#     # ノイズ除去のためにガウシアンフィルタを適用
#     blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
#     # 閾値処理で二値化
#     _, binary = cv2.threshold(blurred, 30, 255, cv2.THRESH_BINARY)

#     # 輪郭を検出
#     contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

#     # 最大の輪郭を取得（乳房領域と仮定）
#     max_contour = max(contours, key=cv2.contourArea)

#     # 境界ボックスを計算
#     x, y, w, h = cv2.boundingRect(max_contour)

#     # 乳房領域を切り出し
#     breast_region = gray[y:y+h, x:x+w]

#     return breast_region

# def process_images(input_dir, output_dir):
#     """
#     ディレクトリ内のすべての画像に対して乳房領域を切り出し
#     """
#     os.makedirs(output_dir, exist_ok=True)

#     for file_name in tqdm(os.listdir(input_dir)):
#         file_path = os.path.join(input_dir, file_name)
        
#         # 画像を読み込み
#         image = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
#         if image is None:
#             print(f"Failed to load image: {file_path}")
#             continue
        
#         # 乳房領域を切り出し
#         breast_region = extract_breast_region(image)

#         # 保存
#         output_path = os.path.join(output_dir, file_name)
#         cv2.imwrite(output_path, breast_region)

# # 入力ディレクトリと出力ディレクトリ
# input_dir = "D:\\Python\\BreastCancer\\before_train2\\1"  # 元の画像フォルダ
# output_dir = "D:\\Python\\BreastCancer\\before_train3\\1"  # 切り出した画像を保存するフォルダ

# # 実行
# process_images(input_dir, output_dir)


# import os
# import pydicom
# from PIL import Image
# import numpy as np
# import glob

# # DICOMからJPGへの変換関数
# def dicom_to_jpg(dicom_path, output_folder, file_index):
#     # DICOMファイルを読み込み
#     dicom_data = pydicom.dcmread(dicom_path)
    
#     # DICOM画像データを取得
#     image_data = dicom_data.pixel_array
    
#     # 画像の正規化（画像のサイズによってはスケーリングを行う）
#     image_data = np.uint8(image_data / np.max(image_data) * 255)
    
#     # 出力先のJPGファイル名（番号順）
#     jpg_filename = f"image_{file_index:04d}.jpg"  # 番号を付けて保存
#     jpg_path = os.path.join(output_folder, jpg_filename)
    
#     # Pillowで画像を保存
#     image = Image.fromarray(image_data)
#     image.save(jpg_path, "JPEG")
#     print(f"Converted {dicom_path} to {jpg_path}")

# # メイン関数
# def convert_dicom_to_jpg(root_dir, output_folder):
#     # 出力先フォルダが存在しない場合は作成
#     if not os.path.exists(output_folder):
#         os.makedirs(output_folder)
    
#     # ワイルドカードを使ってDICOMファイルを検索
#     dicom_files = glob.glob(root_dir, recursive=True)
    
#     # ファイル番号をカウントするための変数
#     file_index = 1
    
#     # すべてのDICOMファイルを処理
#     for dicom_path in dicom_files:
#         if dicom_path.lower().endswith('.dcm'):  # DICOMファイルのみを対象
#             # DICOMファイルをJPGに変換して番号順に保存
#             dicom_to_jpg(dicom_path, output_folder, file_index)
#             file_index += 1  # 次の番号に進む

# # 実行
# root_dir = r"D:\dataset\Mass\Mammograpyh_mass\*\*\*\*.dcm"  # ワイルドカードを使ってDICOMファイルを検索
# # root_dir = r"D:\dataset\Cal\Mammograpyh_math\*\*\*\*.dcm"  # ワイルドカードを使ってDICOMファイルを検索
# output_folder = r"D:\output_jpg_images1"  # JPG画像を保存するフォルダのパス
# convert_dicom_to_jpg(root_dir, output_folder)



timm.list_models(Pretrained=True)

import timm

# 使用可能なモデル一覧を取得
available_models = timm.list_models(pretrained=True)

# 縦に並べて表示
for model in available_models:
    print(model)



cuda:0


model.safetensors:  20%|##        | 52.4M/260M [00:00<?, ?B/s]

C:\Users\user\anaconda3\envs\nn\lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--timm--resnet200d.ra2_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [01:52<00:00, 37.4

Epoch : 1 - loss : 0.6973 - acc : 0.4422 - val_loss : 0.0000 - val_acc : 0.0000



100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [01:25<00:00, 28.55s/it]


KeyboardInterrupt: 